In [1]:
!pip install selenium selenium-stealth fake-useragent webdriver-manager pandas numpy matplotlib seaborn scikit-learn beautifulsoup4 lxml

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

import requests
from bs4 import BeautifulSoup
import time
import random
from fake_useragent import UserAgent
import re
import os

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium_stealth import stealth
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

%matplotlib inline

In [3]:
# Функция для перезапуска драйвера

def restart_driver():
    """Перезапускает WebDriver при ошибке invalid session id"""
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium_stealth import stealth
    from webdriver_manager.chrome import ChromeDriverManager
    from selenium.webdriver.chrome.service import Service
    
    print("ПЕРЕЗАПУСК ДРАЙВЕРА...")
    
    options = Options()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-extensions")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-infobars")
    options.add_argument("--disable-popup-blocking")
    options.add_argument("--start-maximized")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    
    new_driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    
    stealth(new_driver,
        languages=["ru-RU", "ru"],
        vendor="Google Inc.",
        platform="Win32",
        fix_hairline=True,
    )
    
    new_driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    
    print("ДРАЙВЕР ПЕРЕЗАПУЩЕН")
    return new_driver

In [4]:
# Настройка опций браузера для максимальной маскировки
options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--disable-extensions")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-gpu")
options.add_argument("--disable-infobars")
options.add_argument("--disable-popup-blocking")
options.add_argument("--start-maximized")

# Отключение автоматизацию
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)

# Создание драйвера
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

# Маскировка под обычного пользователя
stealth(driver,
    languages=["ru-RU", "ru"],
    vendor="Google Inc.",
    platform="Win32",
    fix_hairline=True,
)

# Дополнительная маскировка через JavaScript
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

# Проверка работы браузера
driver.get("https://auto.drom.ru/")
time.sleep(3)
print("Браузер открыл страницу:", driver.title)

Браузер открыл страницу: Продажа автомобилей в России. Подержанные авто, новые. Купить автомобиль б/у от 9 999 рублей.


In [4]:
# Проверка доступа

test_url = "https://auto.drom.ru/all/"
driver.get(test_url)

time.sleep(4)

html = driver.page_source

if "Вы не робот?" in html:
    print("Нас заблокировали")
else:
    print("Страница загружена успешно")
    
    soup = BeautifulSoup(html, 'html.parser')
    all_blocks = soup.find_all('div')

    count = 0 
    for block in all_blocks:
        if "₽" in str(block):
            count = count + 1 

print("Найдено элементов с ценой:", count)

Страница загружена успешно
Найдено элементов с ценой: 138


In [5]:
def get_car_links(driver, url):
    
    driver.get(url)
    time.sleep(random.uniform(4, 8))
    
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    links = []
    
    # Собирает ссылки из основного списка объявлений
    # Ищет все блоки с классом, содержащим "css-" (основной список)
    for a in soup.find_all('a', href=True):
        href = a.get('href')
        if href and '/moscow/' in href and 'auto.drom.ru/moscow/' in href:
            # Исключаем карусель и служебные ссылки
            if 'from=widgetPremium' not in href and 'page=' not in href:
                links.append(href)
    
    # Если ссылок мало, пробуем дополнительный способ
    if len(links) < 10:
        for a in soup.find_all('a', class_='u41pni0'):
            href = a.get('href')
            if href and '/moscow/' in href:
                links.append(href)
    
    # Удаляем дубликаты
    links = list(set(links))
    
    return links

In [6]:
def parse_car_details(driver, url):
    
    driver.get(url)
    time.sleep(random.uniform(4, 8))
    
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    data = {}
    
    # Марка и модель
    brand_elem = soup.find('a', class_='ht8xsj4')
    if brand_elem:
        data['марка_модель'] = brand_elem.text.strip()
    else:
        data['марка_модель'] = None
    
    # Цена - пробуем несколько способов
    price_elem = None
    
    # Способ 1: стандартный
    price_elem = soup.find('div', {'data-ftid': 'bulletin-price'})
    
    # Способ 2: через класс
    if not price_elem:
        price_elem = soup.find('div', class_='wb9m8q1')
    
    # Способ 3: через span
    if not price_elem:
        price_elem = soup.find('span', {'data-ftid': 'bull_price'})
    
    # Способ 4: поиск по тексту "₽"
    if not price_elem:
        all_spans = soup.find_all('span')
        for span in all_spans:
            if '₽' in span.text:
                price_elem = span
                break
    
    if price_elem:
        price_text = price_elem.text.replace('₽', '').replace(' ', '').replace('\xa0', '').strip()
        # Оставляем только цифры
        price_digits = re.sub(r'[^\d]', '', price_text)
        data['цена'] = int(price_digits) if price_digits else None
    else:
        data['цена'] = None
    
    # Характеристики из таблицы
    specs = soup.find_all('tr', {'data-ftid': True})
    
    for spec in specs:
        th = spec.find('th', {'data-ftid': 'property'})
        td = spec.find('td', {'data-ftid': 'value'})
        
        if th and td:
            key = th.text.strip()
            value = td.text.strip()
            
            if key == 'Год выпуска':
                data['год'] = int(value) if value.isdigit() else None
            elif key == 'Двигатель':
                data['двигатель'] = value
            elif key == 'Мощность':
                power_match = re.search(r'(\d+)', value)
                data['мощность'] = int(power_match.group(1)) if power_match else None
            elif key == 'Коробка передач':
                data['коробка'] = value
            elif key == 'Привод':
                data['привод'] = value
            elif key == 'Цвет':
                data['цвет'] = value
            elif key == 'Пробег':
                km_match = re.search(r'([\d\s]+)', value)
                data['пробег'] = km_match.group(1).strip() if km_match else None
            elif key == 'Владельцы':
                data['владельцы'] = int(value) if value.isdigit() else None
            elif key == 'Руль':
                data['руль'] = value
            elif key == 'Поколение':
                data['поколение'] = value
            elif key == 'Комплектация':
                data['комплектация'] = value
    
    # Город
    city_elem = soup.find('div', {'data-ftid': 'city'})
    if city_elem:
        city_value = city_elem.find('span', {'data-ftid': 'value'})
        data['город'] = city_value.text.strip() if city_value else None
    else:
        data['город'] = None
    
    # Тип кузова
    if data.get('марка_модель'):
        body_keywords = ['седан', 'хэтчбек', 'универсал', 'внедорожник', 'купе', 'кабриолет', 'пикап', 'лимузин', 'минивэн']
        name_lower = data['марка_модель'].lower()
        data['кузов'] = None
        for keyword in body_keywords:
            if keyword in name_lower:
                data['кузов'] = keyword
                break
    
    return data

In [9]:
def parse_cars_detailed(driver, start_page, end_page, batch_name):
    
    all_cars = []
    success_pages = 0
    failed_pages = 0
    max_errors = 7
    
    # Сохранение ссылки не драйвер
    current_driver = driver
    
    for page in range(start_page, end_page + 1):
        print("Обработка страницы", page)
        
        if page == 1:
            url = "https://auto.drom.ru/all/"
        else:
            url = "https://auto.drom.ru/all/?page=" + str(page)
        
        try:
            print("  Собираю ссылки на объявления...")
            links = get_car_links(current_driver, url)
            print("  Найдено ссылок:", len(links))
            
            if len(links) == 0:
                print("  На странице нет ссылок на объявления")
                failed_pages = failed_pages + 1
                if failed_pages >= max_errors:
                    print("  Слишком много пустых страниц - останавливаем")
                    break
                continue
            
            failed_pages = 0
            
            for link in links:
                try:
                    print("    Парсинг:", link)
                    car_data = parse_car_details(current_driver, link)
                    
                    if car_data and car_data.get('цена'):
                        all_cars.append(car_data)
                        print("      Цена:", car_data.get('цена'))
                    else:
                        print("      Ошибка: нет цены")
                    
                    time.sleep(random.uniform(3, 5))
                    
                except Exception as e:
                    error_msg = str(e)
                    print("      Ошибка при парсинге ссылки:", error_msg[:100])
                    
                    # Перезапуск при ошибке
                    if "invalid session id" in error_msg.lower():
                        print("      СЕССИЯ УМЕРЛА, ПЕРЕЗАПУСК ДРАЙВЕРа...")
                        try:
                            current_driver.quit()
                        except:
                            pass
                        
                        # вызов новой функции
                        current_driver = restart_driver()
                        print("      ДРАЙВЕР ПЕРЕЗАПУЩЕН")
                        time.sleep(5)
 
                    
                    time.sleep(random.uniform(5, 10))
                    continue
            
            success_pages = success_pages + 1
            print("  Собрано данных:", len(all_cars))
            
        except Exception as e:
            failed_pages = failed_pages + 1
            error_msg = str(e)
            print("  Ошибка на странице", page, ":", error_msg[:100])
            
            # Перезапуск при ошибке сессии
            if "invalid session id" in error_msg.lower():
                print("  СЕССИЯ УМЕРЛА, ПЕРЕЗАПУСКАЮ ДРАЙВЕР...")
                try:
                    current_driver.quit()
                except:
                    pass
                
                # вызов новой функции
                current_driver = restart_driver()
                print("  ДРАЙВЕР ПЕРЕЗАПУЩЕН")
                failed_pages = 0  # сброс счётчика ошибок
                time.sleep(5)
                continue
            
            if failed_pages >= max_errors:
                print("  Слишком много ошибок - останавливаем")
                break
            
            time.sleep(random.uniform(10, 15))
            continue
        
        if page < end_page:
            pause = random.uniform(10, 20)
            print("Пауза", pause, "секунд...")
            time.sleep(pause)
        
        if page % 10 == 0 and page < end_page:
            print("Дополнительная пауза 30-60 секунд после 10 страниц")
            time.sleep(random.uniform(30, 60))
    
    if len(all_cars) > 0:
        df_batch = pd.DataFrame(all_cars)
        filename = "../data/raw/drom_cars_detailed_" + batch_name + ".csv"
        df_batch.to_csv(filename, index=False)
        print("СОХРАНЕНО:", filename)
        print("Всего записей:", len(df_batch))
        return df_batch
    else:
        print("Нет данных для сохранения")
        return None

In [10]:
print("Начало парсинга 80 страниц")

df_result = parse_cars_detailed(
    driver=driver,
    start_page=1,
    end_page=80,
    batch_name='80pages'
)

print("Парсинг завершен")

Начало парсинга 80 страниц
Обработка страницы 1
  Собираю ссылки на объявления...
  Найдено ссылок: 26
    Парсинг: https://auto.drom.ru/moscow/mercedes-benz/gls-class/291652043.html
      Цена: 28490000
    Парсинг: https://auto.drom.ru/moscow/bmw/x7/358790413.html
      Цена: 17750000
    Парсинг: https://auto.drom.ru/moscow/all/?distance=1000
      Цена: 12800000
    Парсинг: https://auto.drom.ru/moscow/all/?distance=100
      Цена: 12800000
    Парсинг: https://auto.drom.ru/moscow/mercedes-benz/s-class/179587669.html
      Цена: 14990000
    Парсинг: https://auto.drom.ru/moscow/bmw/x7/579728771.html
      Цена: 16950000
    Парсинг: https://auto.drom.ru/moscow/toyota/tacoma/858727055.html
      Цена: 11490000
    Парсинг: https://auto.drom.ru/moscow/bmw/x7/350910767.html
      Цена: 11990000
    Парсинг: https://auto.drom.ru/moscow/all/?distance=500
      Цена: 12800000
    Парсинг: https://auto.drom.ru/moscow/bmw/x7/413593812.html
      Цена: 10490000
    Парсинг: https://auto.dro

In [11]:
print("Собрано записей:", len(df_result) if df_result is not None else 0)

if df_result is not None:
    print("Колонки:", df_result.columns.tolist())
    print("Первые 5 записей:")
    display(df_result.head())

Собрано записей: 1084
Колонки: ['марка_модель', 'цена', 'год', 'двигатель', 'мощность', 'коробка', 'привод', 'цвет', 'пробег', 'руль', 'поколение', 'комплектация', 'город', 'кузов', 'владельцы']
Первые 5 записей:


,марка_модель,цена,год,двигатель,мощность,коробка,привод,цвет,пробег,руль,поколение,комплектация,город,кузов,владельцы
0,Мерседес GLS-класс,28490000,2026.0,"бензин, 4.0 л, гибрид",557.0,АКПП,4WD,черный,81,левый,"2 поколение, рестайлинг",Mercedes-Maybach GLS 600 4MATIC,Москва,NaN,NaN
1,БМВ Х7,17750000,2025.0,"бензин, 4.4 л, гибрид",530.0,АКПП,4WD,белый,1 250,левый,"1 поколение, рестайлинг",M60i AT M Sport,Москва,NaN,NaN
2,None,12800000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN
3,None,12800000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN
4,Мерседес S-Класс,14990000,2023.0,"бензин, 3.0 л, гибрид",367.0,АКПП,4WD,белый,1 129,левый,7 поколение,S580 e 4MATIC L,Москва,NaN,NaN


In [13]:
print("Перезапускаю драйвер...")

# Закрываем старый
try:
    driver.quit()
except:
    print("Старый драйвер уже закрыт")

# Создаем новый
options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_experimental_option("excludeSwitches", ["enable-automation"])

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)
driver.set_page_load_timeout(180)

stealth(driver,
    languages=["ru-RU", "ru"],
    vendor="Google Inc.",
    platform="Win32",
    fix_hairline=True,
)

print("Драйвер перезапущен")

Перезапускаю драйвер...
Драйвер перезапущен


In [14]:
# Продолжение парсинга 

# Путь где уже есть 1084 записи
file_name = "../data/raw/drom_cars_detailed_80pages.csv"

# Загрузка существующих данных
print("Загружаю существующий файл...")
df_existing = pd.read_csv(file_name)
print("В файле уже есть", len(df_existing), "записей")

# Список для всех записей (старые + новые)
all_cars = df_existing.to_dict('records')

# Парсинг с 45 по 80 стр
for page in range(45, 81):
    print("Обрабатка страницы", page)
    
    # Формируем URL
    if page == 1:
        url = "https://auto.drom.ru/all/"
    else:
        url = "https://auto.drom.ru/all/?page=" + str(page)
    
    try:
        # Сбор ссылки на объявления
        print("Собираю ссылки...")
        links = get_car_links(driver, url)
        print("Найдено ссылок:", len(links))
        
        # Если ссылок нет - пропуск
        if len(links) == 0:
            print("На странице нет объявлений")
            continue
        
        # Парсим каждую ссылку
        for link in links:
            try:
                print("Парсинг:", link[:60], "...")
                
                # Данные об авто
                car_data = parse_car_details(driver, link)
                
                # Если есть цена - сохраняем
                if car_data and car_data.get('цена'):
                    all_cars.append(car_data)
                    print("      Цена:", car_data.get('цена'), "руб.")
                    print("      Всего записей теперь:", len(all_cars))
                else:
                    print("      Нет цены, пропускаем")
                
                # Пауза между объявлениями
                time.sleep(random.uniform(2, 4))
                
            except Exception as e:
                print("      Ошибка при парсинге:", str(e)[:50])
                
                # Если ошибка с сессией - перезапуск драйвера
                if "invalid session" in str(e).lower() or "timeout" in str(e).lower():
                    print("      Перезапускаю драйвер...")
                    try:
                        driver.quit()
                    except:
                        pass
                    
                    # Создаение нового драйвера
                    options = Options()
                    options.add_argument("--disable-blink-features=AutomationControlled")
                    options.add_argument("--no-sandbox")
                    options.add_argument("--disable-dev-shm-usage")
                    options.add_argument("--window-size=1920,1080")
                    options.add_experimental_option("excludeSwitches", ["enable-automation"])
                    
                    driver = webdriver.Chrome(
                        service=Service(ChromeDriverManager().install()),
                        options=options
                    )
                    driver.set_page_load_timeout(180)
                    
                    stealth(driver,
                        languages=["ru-RU", "ru"],
                        vendor="Google Inc.",
                        platform="Win32",
                        fix_hairline=True,
                    )
                    
                    print("Драйвер перезапущен")
                    time.sleep(5)
                
                time.sleep(random.uniform(3, 5))
                continue
        
        # Сохраняем прогресс каждые 50 новых записей
        if len(all_cars) % 50 == 0:
            df_temp = pd.DataFrame(all_cars)
            df_temp.to_csv(file_name, index=False, encoding='utf-8-sig')
            print("  ПРОМЕЖУТОЧНОЕ СОХРАНЕНИЕ:", len(all_cars), "записей")
        
    except Exception as e:
        print("  Ошибка на странице", page, ":", str(e)[:80])
        print("  Пропускаю страницу и продолжаю...")
        time.sleep(random.uniform(5, 10))
        continue
    
    # Пауза между страницами
    pause_time = random.uniform(8, 15)
    print("  Пауза", round(pause_time, 1), "секунд...")
    time.sleep(pause_time)
    
    # Дополнительная пауза каждые 10 страниц
    if page % 10 == 0:
        print("Дополнительная пауза каждые 30 секунд")
        time.sleep(30)

# Финальное сохранение
df_final = pd.DataFrame(all_cars)
df_final.to_csv(file_name, index=False, encoding='utf-8-sig')

print("Всего записей:", len(df_final))
print("Файл сохранен:", file_name)
print("")
print("Первые 5 записей:")
display(df_final.head())

Загружаю существующий файл...
В файле уже есть 1084 записей
Обрабатка страницы 45
Собираю ссылки...
Найдено ссылок: 26
Парсинг: https://auto.drom.ru/moscow/lexus/lx700h/741661614.html ...
      Цена: 19285000 руб.
      Всего записей теперь: 1085
Парсинг: https://auto.drom.ru/moscow/all/?distance=1000 ...
      Цена: 2800000 руб.
      Всего записей теперь: 1086
Парсинг: https://auto.drom.ru/moscow/all/?distance=100 ...
      Цена: 930000 руб.
      Всего записей теперь: 1087
Парсинг: https://auto.drom.ru/moscow/bmw/5-series/738992676.html ...
      Цена: 3900000 руб.
      Всего записей теперь: 1088
Парсинг: https://auto.drom.ru/moscow/zeekr/9x/136194631.html ...
      Цена: 11299000 руб.
      Всего записей теперь: 1089
Парсинг: https://auto.drom.ru/moscow/all/?distance=500 ...
      Цена: 930000 руб.
      Всего записей теперь: 1090
Парсинг: https://auto.drom.ru/moscow/nissan/pathfinder/439884100.html ...
      Цена: 1150000 руб.
      Всего записей теперь: 1091
Парсинг: https://aut

,марка_модель,цена,год,двигатель,мощность,коробка,привод,цвет,пробег,руль,поколение,комплектация,город,кузов,владельцы
0,Мерседес GLS-класс,28490000,2026.0,"бензин, 4.0 л, гибрид",557.0,АКПП,4WD,черный,81,левый,"2 поколение, рестайлинг",Mercedes-Maybach GLS 600 4MATIC,Москва,NaN,NaN
1,БМВ Х7,17750000,2025.0,"бензин, 4.4 л, гибрид",530.0,АКПП,4WD,белый,1 250,левый,"1 поколение, рестайлинг",M60i AT M Sport,Москва,NaN,NaN
2,NaN,12800000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,12800000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Мерседес S-Класс,14990000,2023.0,"бензин, 3.0 л, гибрид",367.0,АКПП,4WD,белый,1 129,левый,7 поколение,S580 e 4MATIC L,Москва,NaN,NaN


In [15]:
# Продолжение парсинга 

# Путь где уже есть 1984 записи
file_name = "../data/raw/drom_cars_detailed_80pages.csv"

# Загрузка существующих данных
print("Загружаю существующий файл...")
df_existing = pd.read_csv(file_name)
print("В файле уже есть", len(df_existing), "записей")

# Список для всех записей (старые + новые)
all_cars = df_existing.to_dict('records')

# Парсинг с 81 по 200 стр
for page in range(81, 201):
    print("Обрабатка страницы", page)
    
    # Формируем URL
    if page == 1:
        url = "https://auto.drom.ru/all/"
    else:
        url = "https://auto.drom.ru/all/?page=" + str(page)
    
    try:
        # Сбор ссылки на объявления
        print("Собираю ссылки...")
        links = get_car_links(driver, url)
        print("Найдено ссылок:", len(links))
        
        # Если ссылок нет - пропуск
        if len(links) == 0:
            print("На странице нет объявлений")
            continue
        
        # Парсим каждую ссылку
        for link in links:
            try:
                print("Парсинг:", link[:60], "...")
                
                # Данные об авто
                car_data = parse_car_details(driver, link)
                
                # Если есть цена - сохраняем
                if car_data and car_data.get('цена'):
                    all_cars.append(car_data)
                    print("      Цена:", car_data.get('цена'), "руб.")
                    print("      Всего записей теперь:", len(all_cars))
                else:
                    print("      Нет цены, пропускаем")
                
                # Пауза между объявлениями
                time.sleep(random.uniform(2, 4))
                
            except Exception as e:
                print("      Ошибка при парсинге:", str(e)[:50])
                
                # Если ошибка с сессией - перезапуск драйвера
                if "invalid session" in str(e).lower() or "timeout" in str(e).lower():
                    print("      Перезапускаю драйвер...")
                    try:
                        driver.quit()
                    except:
                        pass
                    
                    # Создаение нового драйвера
                    options = Options()
                    options.add_argument("--disable-blink-features=AutomationControlled")
                    options.add_argument("--no-sandbox")
                    options.add_argument("--disable-dev-shm-usage")
                    options.add_argument("--window-size=1920,1080")
                    options.add_experimental_option("excludeSwitches", ["enable-automation"])
                    
                    driver = webdriver.Chrome(
                        service=Service(ChromeDriverManager().install()),
                        options=options
                    )
                    driver.set_page_load_timeout(180)
                    
                    stealth(driver,
                        languages=["ru-RU", "ru"],
                        vendor="Google Inc.",
                        platform="Win32",
                        fix_hairline=True,
                    )
                    
                    print("Драйвер перезапущен")
                    time.sleep(5)
                
                time.sleep(random.uniform(3, 5))
                continue
        
        # Сохраняем прогресс каждые 50 новых записей
        if len(all_cars) % 50 == 0:
            df_temp = pd.DataFrame(all_cars)
            df_temp.to_csv(file_name, index=False, encoding='utf-8-sig')
            print("  ПРОМЕЖУТОЧНОЕ СОХРАНЕНИЕ:", len(all_cars), "записей")
        
    except Exception as e:
        print("  Ошибка на странице", page, ":", str(e)[:80])
        print("  Пропускаю страницу и продолжаю...")
        time.sleep(random.uniform(5, 10))
        continue
    
    # Пауза между страницами
    pause_time = random.uniform(8, 15)
    print("  Пауза", round(pause_time, 1), "секунд...")
    time.sleep(pause_time)
    
    # Дополнительная пауза каждые 10 страниц
    if page % 10 == 0:
        print("Дополнительная пауза каждые 30 секунд")
        time.sleep(30)

# Финальное сохранение
df_final = pd.DataFrame(all_cars)
df_final.to_csv(file_name, index=False, encoding='utf-8-sig')

print("Всего записей:", len(df_final))
print("Файл сохранен:", file_name)
print("")
print("Первые 5 записей:")
display(df_final.head())

Загружаю существующий файл...
В файле уже есть 1984 записей
Обрабатка страницы 81
Собираю ссылки...
Найдено ссылок: 26
Парсинг: https://auto.drom.ru/moscow/lexus/lx700h/741661614.html ...
      Цена: 19285000 руб.
      Всего записей теперь: 1985
Парсинг: https://auto.drom.ru/moscow/all/?distance=1000 ...
      Цена: 799000 руб.
      Всего записей теперь: 1986
Парсинг: https://auto.drom.ru/moscow/all/?distance=100 ...
      Цена: 1630000 руб.
      Всего записей теперь: 1987
Парсинг: https://auto.drom.ru/moscow/bmw/5-series/738992676.html ...
      Цена: 3900000 руб.
      Всего записей теперь: 1988
Парсинг: https://auto.drom.ru/moscow/zeekr/9x/136194631.html ...
      Цена: 11299000 руб.
      Всего записей теперь: 1989
Парсинг: https://auto.drom.ru/moscow/all/?distance=500 ...
      Цена: 97000 руб.
      Всего записей теперь: 1990
Парсинг: https://auto.drom.ru/moscow/nissan/pathfinder/439884100.html ...
      Цена: 1150000 руб.
      Всего записей теперь: 1991
Парсинг: https://auto

,марка_модель,цена,год,двигатель,мощность,коробка,привод,цвет,пробег,руль,поколение,комплектация,город,кузов,владельцы
0,Мерседес GLS-класс,28490000,2026.0,"бензин, 4.0 л, гибрид",557.0,АКПП,4WD,черный,81,левый,"2 поколение, рестайлинг",Mercedes-Maybach GLS 600 4MATIC,Москва,NaN,NaN
1,БМВ Х7,17750000,2025.0,"бензин, 4.4 л, гибрид",530.0,АКПП,4WD,белый,1 250,левый,"1 поколение, рестайлинг",M60i AT M Sport,Москва,NaN,NaN
2,NaN,12800000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,12800000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Мерседес S-Класс,14990000,2023.0,"бензин, 3.0 л, гибрид",367.0,АКПП,4WD,белый,1 129,левый,7 поколение,S580 e 4MATIC L,Москва,NaN,NaN
